In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
sns.set_style('darkgrid')
df = pd.read_csv("..\\datasets\\mercc_sessid.csv")
df.head()

,user_id,stime,session_id,sequence_id,sequence_length,event_id,item_id,product_id,name,price,...,c2_id,brand_name,brand_id,item_condition_id,item_condition_name,size_name,size_id,color,shipper_id,shipper_name
0,1711,2023-05-28 17:03:55,9c06d0a6c644d16c2f88bd40f1d259a255570c6505c45b...,20230501_3,9,item_view,51045461,4054_1332,Chantecaille Rouge Perle AKOYA,80.0,...,1332.0,Chantecaille,4054,2,Like new,NaN,NaN,NaN,1,Buyer
1,1711,2023-05-28 17:05:12,9c06d0a6c644d16c2f88bd40f1d259a255570c6505c45b...,20230501_3,9,item_view,219171710,21591_1332,Surratt Artistique Blush PARFAIT,19.0,...,1332.0,SURRATT BEAUTY,21591,3,Good,NaN,NaN,NaN,1,Buyer
2,1711,2023-05-28 17:09:49,9c06d0a6c644d16c2f88bd40f1d259a255570c6505c45b...,20230501_3,9,item_view,152031642,19885_2338,SLIP SILK The Icons Edit,75.0,...,2338.0,Slip,19885,1,New,NaN,NaN,NaN,1,Buyer
3,1711,2023-05-28 17:11:29,9c06d0a6c644d16c2f88bd40f1d259a255570c6505c45b...,20230501_3,9,item_view,51045461,4054_1332,Chantecaille Rouge Perle AKOYA,80.0,...,1332.0,Chantecaille,4054,2,Like new,NaN,NaN,NaN,1,Buyer
4,1711,2023-05-28 17:15:23,9c06d0a6c644d16c2f88bd40f1d259a255570c6505c45b...,20230501_3,9,item_view,125409152,802_338,Apple airpod - generation 2 - left only,26.0,...,338.0,Apple,802,3,Good,NaN,NaN,other,1,Buyer


In [3]:
categ = pd.read_csv('..\\datasets\\brandnames.csv')

mapping_dict = dict(zip(categ['brand'], categ['Category']))
df['brand_category'] = df['brand_name'].map(mapping_dict)
df['brand_category'] = df['brand_category'].fillna('Uncategorized')

In [4]:
# Transforming datatypes
df.drop(['shipper_id', 'shipper_name'], axis=1)

df[['c1_id', 'c2_id', 'size_id']] = df[['c1_id', 'c2_id', 'size_id']].fillna(0)
df[['c1_name', 'c2_name', 'size_name', 'brand_name', 'color', 'item_condition_name']] = df[['c1_name', 'c2_name', 'size_name', 'brand_name', 'color', 'item_condition_name']].fillna('unknown')

df = df.astype({
    'c1_id':'int32',
    'c2_id':'int32',
    'size_id':'int32',
    'c1_name':'category',
    'c2_name':'category',
    'size_name':'category',
    'item_condition_name':'category',
    'event_id':'category'
})

# Dtypes optimization
df['stime'] = pd.to_datetime(df['stime'])
df['item_condition_name'] = df['item_condition_name'].cat.reorder_categories(['New', 'Like new', 'Good', 'Fair', 'Poor', 'unknown'])
df['event_id'] = df['event_id'].cat.reorder_categories(['item_view', 'item_like', 'item_add_to_cart_tap', 'buy_start', 'offer_make', 'buy_comp'])

df = df.drop_duplicates(subset=['session_id', 'item_id', 'event_id']) # Data deduplication

# Data Categorization
min_sess_time = df.groupby('session_id')['stime'].transform('min')
max_sess_time = df.groupby('session_id')['stime'].transform('max')

df['session_length'] = max_sess_time - min_sess_time
df['session_length_min'] = df['session_length'].dt.total_seconds()/60
df['session_length_min'].agg(['median','std', 'min','max'])

# Feature Scaling
df['log_price'] = np.log(df['price'])

In [5]:
"""
conditions = [
    (df['session_length_min'] <= 5),
    (df['session_length_min'] > 5) & (df['session_length_min'] <= 20),
    (df['session_length_min'] > 20) & (df['session_length_min'] <= 60),
    (df['session_length_min'] > 60)
]
results = ['Instant shoppers', 'Low-intensity shoppers','High-intensity shoppers','Inactive/campers shoppers']

df['shopper_category'] = np.select(conditions, results, default=None)
"""

"""
    To do: add sequence length to the conditions in categorizing customers time spent
"""

'\n    To do: add sequence length to the conditions in categorizing customers time spent\n'

In [6]:
# This line took the longest time
#df.to_csv('datasets\\mercc_cleaned.csv')

In [7]:
# dtypes inspections
df.dtypes

user_id                          int64
stime                   datetime64[us]
session_id                         str
sequence_id                        str
sequence_length                  int64
event_id                      category
item_id                          int64
product_id                         str
name                               str
price                          float64
c0_name                            str
c0_id                            int64
c1_name                       category
c1_id                            int32
c2_name                       category
c2_id                            int32
brand_name                         str
brand_id                         int64
item_condition_id                int64
item_condition_name           category
size_name                     category
size_id                          int32
color                              str
shipper_id                       int64
shipper_name                       str
brand_category           

In [8]:
# Data columns
df.columns

Index(['user_id', 'stime', 'session_id', 'sequence_id', 'sequence_length',
       'event_id', 'item_id', 'product_id', 'name', 'price', 'c0_name',
       'c0_id', 'c1_name', 'c1_id', 'c2_name', 'c2_id', 'brand_name',
       'brand_id', 'item_condition_id', 'item_condition_name', 'size_name',
       'size_id', 'color', 'shipper_id', 'shipper_name', 'brand_category',
       'session_length', 'session_length_min', 'log_price'],
      dtype='str')